Prueba

In [1]:
pip install hyperopt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [3]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\OneDrive\\Escritorio\\colab\\SNconsumptionFinal.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [4]:
datos.head()

,temp,zone1,zone2,zone3,hour
date,,,,,
2017-01-01 00:00:00,6.559,34055.69620,16128.87538,20240.96386,0
2017-01-01 00:10:00,6.414,29814.68354,19375.07599,20131.08434,0
2017-01-01 00:20:00,6.313,29128.10127,19006.68693,19668.43373,0
2017-01-01 00:30:00,6.121,28228.86076,18361.09422,18899.27711,0
2017-01-01 00:40:00,5.921,27335.69620,17872.34043,18442.40964,0


In [5]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Se divide el dataset

In [6]:
# Dividir el conjunto de datos en entrenamiento y prueba
train, test = train_test_split(datos, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
test, val = train_test_split(test, test_size=0.33, shuffle=False)

print("Las dimensiones de train son: ", train.shape)
print("Las dimensiones de test son: ", test.shape)
print("Las dimensiones de val son: ", val.shape)

Las dimensiones de train son:  (36691, 5)
Las dimensiones de test son:  (10535, 5)
Las dimensiones de val son:  (5190, 5)


Se normalizan los datos

In [7]:
from sklearn.preprocessing import StandardScaler


# Normalizar solo con los datos de entrenamiento
scaler = StandardScaler()
train = scaler.fit_transform(train)

# Aplicar la transformación a test y val usando los parámetros de train
test = scaler.transform(test)
val = scaler.transform(val)

train = pd.DataFrame(train, columns=datos.columns)
test = pd.DataFrame(test, columns=datos.columns)
val = pd.DataFrame(val, columns=datos.columns)


Se unen los datos nuevamente, ahora normalizados, en un único conjunto.

In [8]:
datosNormalizados = pd.concat([train, test, val])

datosNormalizados.index = datos.index


In [9]:
datosNormalizados.shape

(52416, 5)

In [10]:
datosNormalizados.head(10)


,temp,zone1,zone2,zone3,hour
date,,,,,
2017-01-01 00:00:00,-2.051356,0.151445,-0.851669,0.033500,-1.660858
2017-01-01 00:10:00,-2.074813,-0.433079,-0.211097,0.016502,-1.660858
2017-01-01 00:20:00,-2.091152,-0.527709,-0.283791,-0.055068,-1.660858
2017-01-01 00:30:00,-2.122213,-0.651648,-0.411186,-0.174054,-1.660858
2017-01-01 00:40:00,-2.154568,-0.774750,-0.507632,-0.244729,-1.660858
2017-01-01 00:50:00,-2.165569,-0.872729,-0.597600,-0.293039,-1.660858
2017-01-01 01:00:00,-2.199865,-0.958984,-0.681090,-0.321667,-1.516340
2017-01-01 01:10:00,-2.223323,-1.035189,-0.746587,-0.396816,-1.516340
2017-01-01 01:20:00,-2.193880,-1.127306,-0.832236,-0.463913,-1.516340


Espacio de búsqueda

In [11]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'epochs': hp.choice('epochs', [2 ** i for i in range(3, 9)]),  # Número de épocas de entrenamiento
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [12]:
futuros = 24
pasados  = 12

In [13]:
datosX = []
datosY = []
for i in range(pasados, len(datosNormalizados) - futuros + 1):
  datosX.append(datosNormalizados.iloc[i-pasados:i, 0:datosNormalizados.shape[1]])
  datosY.append(datosNormalizados.iloc[i+futuros-1:i+futuros, 1])


In [14]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (52381, 12, 5)
Dimensiones de Y: (52381, 1)


In [15]:
inputs = datosX.shape[1] * datosX.shape[2]
datosX = datosX.reshape(datosX.shape[0], inputs)

In [16]:
print("Dimensiones de X después de rehape:", datosX.shape)

Dimensiones de X después de rehape: (52381, 60)


Se dividen nuevamente los conjuntos de datos

In [17]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (36666, 60)
Las dimensiones de testX son:  (10529, 60)
Las dimensiones de valX son:  (5186, 60)


In [18]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (36666, 1)
Las dimensiones de testY son:  (10529, 1)
Las dimensiones de valY son:  (5186, 1)


Se crean métricas para medir desempeño

In [19]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [20]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [21]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(trainX.shape[1],)))
    if (params['layers'] == 1):
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(Dense(units=params['units'], activation=params['activation']))
          model.add(Dropout(params['dropout']))
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(trainX, trainY, epochs=params['epochs'],
                        validation_data=(testX, testY),
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [22]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=50, trials=trials, rstate=np.random.default_rng(42))

print(best)

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

1146/1146 - 25s - 22ms/step - ia: 0.3337 - loss: 1.3041 - mae: 0.8756 - rmse: 1.0993 - smape: 1.4130 - val_ia: 0.2520 - val_loss: 0.9318 - val_mae: 0.7932 - val_rmse: 0.8629 - val_smape: 1.4030

Epoch 2/128                                           

1146/1146 - 9s - 8ms/step - ia: 0.4165 - loss: 0.7816 - mae: 0.7118 - rmse: 0.8778 - smape: 1.3042 - val_ia: 0.2514 - val_loss: 1.1205 - val_mae: 0.8785 - val_rmse: 0.9476 - val_smape: 1.3789

Epoch 3/128                                           

1146/1146 - 5s - 4ms/step - ia: 0.4652 - loss: 0.6911 - mae: 0.6688 - rmse: 0.8258 - smape: 1.2183 - val_ia: 0.2449 - val_loss: 1.2003 - val_mae: 0.9178 - val_rmse: 0.9849 - val_smape: 1.3899

Epoch 4/128                                           

1146/1146 - 7s - 6ms/step - ia: 0.4865 - loss: 0.6516 - mae: 0.6492 - rmse: 0.8018 - smape: 1.1756 - val_ia: 0.2407 - val_loss: 1.2256 - val_mae: 0.9279 - val_rmse: 0.9961 - val_smape: 1.3840

Ep

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/16                                                                         

144/144 - 8s - 57ms/step - ia: 0.6949 - loss: 0.3147 - mae: 0.4293 - rmse: 0.5412 - smape: 0.8346 - val_ia: 0.5672 - val_loss: 0.6156 - val_mae: 0.6257 - val_rmse: 0.7591 - val_smape: 1.1070

Epoch 2/16                                                                         

144/144 - 2s - 12ms/step - ia: 0.8205 - loss: 0.1401 - mae: 0.2830 - rmse: 0.3731 - smape: 0.6285 - val_ia: 0.6248 - val_loss: 0.4358 - val_mae: 0.5078 - val_rmse: 0.6333 - val_smape: 1.0256

Epoch 3/16                                                                         

144/144 - 1s - 4ms/step - ia: 0.8458 - loss: 0.1078 - mae: 0.2461 - rmse: 0.3278 - smape: 0.5702 - val_ia: 0.6653 - val_loss: 0.3279 - val_mae: 0.4285 - val_rmse: 0.5511 - val_smape: 0.8946

Epoch 4/16                                                                         

144/144 - 1s - 5ms/step - ia: 0.8581 - loss: 0.0930 - mae: 0.2277 - rmse: 0.3042 - sm

...

Epoch 1/16                                                                           

573/573 - 17s - 30ms/step - ia: 0.5526 - loss: 0.6600 - mae: 0.6544 - rmse: 0.8082 - smape: 1.0728 - val_ia: 0.3497 - val_loss: 1.2860 - val_mae: 0.9781 - val_rmse: 1.0682 - val_smape: 1.3790

Epoch 2/16                                                                           

573/573 - 2s - 3ms/step - ia: 0.6045 - loss: 0.5201 - mae: 0.5800 - rmse: 0.7183 - smape: 0.9824 - val_ia: 0.3620 - val_loss: 1.2688 - val_mae: 0.9489 - val_rmse: 1.0478 - val_smape: 1.3407

Epoch 3/16                                                                           

573/573 - 1s - 2ms/step - ia: 0.6256 - loss: 0.4701 - mae: 0.5493 - rmse: 0.6832 - smape: 0.9482 - val_ia: 0.3667 - val_loss: 1.3070 - val_mae: 0.9508 - val_rmse: 1.0567 - val_smape: 1.3243

Epoch 4/16                                                                           

573/573 - 2s - 3ms/step - ia: 0.6397 - loss: 0.4390 - mae: 0.5306 - rmse: 0.6

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



4584/4584 - 17s - 4ms/step - ia: 0.5608 - loss: 0.5408 - mae: 0.5841 - rmse: 0.7100 - smape: 0.9960 - val_ia: 0.1048 - val_loss: 1.6775 - val_mae: 1.1147 - val_rmse: 1.1348 - val_smape: 1.4218

Epoch 2/128                                                                          

4584/4584 - 22s - 5ms/step - ia: 0.5689 - loss: 0.5042 - mae: 0.5654 - rmse: 0.6886 - smape: 0.9806 - val_ia: 0.1057 - val_loss: 1.4517 - val_mae: 1.0414 - val_rmse: 1.0605 - val_smape: 1.4229

Epoch 3/128                                                                          

4584/4584 - 11s - 2ms/step - ia: 0.5693 - loss: 0.5031 - mae: 0.5652 - rmse: 0.6881 - smape: 0.9791 - val_ia: 0.0988 - val_loss: 1.7742 - val_mae: 1.1425 - val_rmse: 1.1674 - val_smape: 1.3918

Epoch 4/128                                                                          

4584/4584 - 22s - 5ms/step - ia: 0.5703 - loss: 0.5009 - mae: 0.5637 - rmse: 0.6861 - smape: 0.9793 - val_ia: 0.1129 - val_loss: 1.2725 - val_mae: 0.9608 - v

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



287/287 - 4s - 15ms/step - ia: 0.5230 - loss: 0.7106 - mae: 0.6697 - rmse: 0.8338 - smape: 1.1157 - val_ia: 0.4128 - val_loss: 1.0961 - val_mae: 0.8598 - val_rmse: 1.0179 - val_smape: 1.3102

Epoch 2/32                                                                           

287/287 - 2s - 7ms/step - ia: 0.5849 - loss: 0.5297 - mae: 0.5772 - rmse: 0.7264 - smape: 1.0015 - val_ia: 0.4256 - val_loss: 1.0065 - val_mae: 0.8092 - val_rmse: 0.9763 - val_smape: 1.2788

Epoch 3/32                                                                           

287/287 - 1s - 3ms/step - ia: 0.5947 - loss: 0.5092 - mae: 0.5662 - rmse: 0.7122 - smape: 0.9872 - val_ia: 0.4187 - val_loss: 1.0391 - val_mae: 0.8231 - val_rmse: 0.9915 - val_smape: 1.2859

Epoch 4/32                                                                           

287/287 - 1s - 3ms/step - ia: 0.5985 - loss: 0.5030 - mae: 0.5612 - rmse: 0.7078 - smape: 0.9795 - val_ia: 0.4210 - val_loss: 1.0650 - val_mae: 0.8299 - val_rmse: 1.

In [23]:
print(best)

{'activation': 1, 'batch': 1, 'dropout': 0.30000000000000004, 'epochs': 4, 'layers': 1.0, 'learning_rate': 0.008417176284345648, 'units': 4}
